**Import core libraries**: pandas/numpy for data handling, and MinMaxScaler from scikit-learn for feature normalization.

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler

**Load data**
Load the mobile phone dataset from Dataset.xlsx into a DataFrame df.

In [4]:
df = pd.read_excel('Dataset.xlsx')

**Price segmentation**

Bucket each phone into a price tier based on Price:

< 10000 → Budget

10000–19999 → Mid Range

20000–34999 → Premium

≥ 35000 → Ultra Premium

Result stored in new column price_segment.

In [5]:
def price_segment(price):
    if price < 10000:
        return "Budget"
    elif price < 20000:
        return "Mid Range"
    elif price < 35000:
        return "Premium"
    else:
        return "Ultra Premium"

df["price_segment"] = df["Price"].apply(price_segment)

 **Price per GB**

Compute price_per_gb = Price / Storage, a normalized cost metric showing how much each GB of storage costs on a given phone (lower = better value on storage).

In [6]:
df["price_per_gb"] = df["Price"] / df["Storage"]

**Battery-to-price ratio**

Compute battery_price_ratio = Battery / Price, showing battery capacity (mAh) obtained per rupee spent (higher = better battery value).

In [7]:
df["battery_price_ratio"] = df["Battery"] / df["Price"]

**Performance score**

Min-max scale RAM, Storage, Battery, and Rating to a 0–1 range, then combine into a weighted performance_score:

RAM: 40%
Storage: 30%
Battery: 20%
Rating: 10%

This gives a single composite metric of overall hardware + user-rated performance.

In [8]:
cols = ["RAM", "Storage", "Battery", "Rating"]

scaler = MinMaxScaler()

scaled = scaler.fit_transform(df[cols])

scaled = pd.DataFrame(scaled, columns=cols)

df["performance_score"] = (
    scaled["RAM"] * 0.4 +
    scaled["Storage"] * 0.3 +
    scaled["Battery"] * 0.2 +
    scaled["Rating"] * 0.1
)

**Value score**

Compute value_score = (performance_score × Rating) / log(1 + Price).
Combines performance and rating as the "benefit" numerator, and uses log-scaled price as the "cost" denominator (log dampens the effect of very high prices) — a rough performance-for-money metric.

In [9]:
df["value_score"] = (
    df["performance_score"] *
    df["Rating"]
) / np.log1p(df["Price"])

**Discount percentage**

Compute discount_pct = Discount / (Price + Discount) × 100.
Expresses the discount as a percentage of the original (pre-discount) price.

In [10]:
df["discount_pct"] = (
    df["Discount"] / (df["Price"] + df["Discount"])
) * 100

 **RAM per ₹1000**

Compute ram_per_1000 = RAM / (Price / 1000), i.e., how much RAM (GB) you get per ₹1000 spent.

In [11]:
df["ram_per_1000"] = df["RAM"] / (df["Price"] / 1000)

**Storage per ₹1000**

Compute storage_per_1000 = Storage / (Price / 1000), i.e., how much storage (GB) you get per ₹1000 spent.

In [12]:
df["storage_per_1000"] = df["Storage"] / (df["Price"] / 1000)

 **Battery per ₹1000**

Compute battery_per_1000 = Battery / (Price / 1000), i.e., how much battery capacity (mAh) you get per ₹1000 spent.

In [13]:
df["battery_per_1000"] = df["Battery"] / (df["Price"] / 1000)

 **Premium brand flag**

Create a binary flag premium_brand: 1 if the phone's Brand is Apple, Samsung, or Google; 0 otherwise. Useful as a categorical feature for modeling brand-driven price/perception effects.

In [14]:
premium = ["Apple", "Samsung", "Google"]
df["premium_brand"] = df["Brand"].isin(premium).astype(int)

 **Hardware score**

Min-max scale RAM, Storage, and Battery again (separately from Cell 6, since Rating is excluded this time), then combine into hardware_score:

RAM: 50%
Storage: 30%
Battery: 20%

A pure hardware-capability score, independent of user rating or price — distinct from performance_score 

In [15]:
cols = ["RAM", "Storage", "Battery"]

scaler = MinMaxScaler()

scaled = pd.DataFrame(
    scaler.fit_transform(df[cols]),
    columns=cols,
    index=df.index
)

df["hardware_score"] = (
    scaled["RAM"] * 0.5 +
    scaled["Storage"] * 0.3 +
    scaled["Battery"] * 0.2
)

In [16]:
#  df.to_excel('output.xlsx', index=False)

In [17]:
# Display Size Extraction

def extract_display_size(display):
    match = re.search(r"\(([\d.]+)\s*inch", str(display))
    if match:
        return float(match.group(1))
    return np.nan

df["Display_Size"] = df["Display"].apply(extract_display_size)

In [18]:
# Display Type Extraction

def extract_display_type(display):

    display = str(display).upper()

    if "AMOLED" in display:
        return "AMOLED"

    elif "OLED" in display:
        return "OLED"

    elif "LCD" in display:
        return "LCD"

    elif "IPS" in display:
        return "IPS"

    else:
        return "Other"

df["Display_Type"] = df["Display"].apply(extract_display_type)

In [19]:
# Rear Camera Extraction

def extract_rear_camera(camera):

    camera = str(camera)

    rear = camera.split("|")[0]

    values = re.findall(r"(\d+)MP", rear)

    if values:
        return int(values[0])

    return np.nan

df["Rear_Main_MP"] = df["Camera"].apply(extract_rear_camera)


In [20]:
# Front Camera Extraction

def extract_front_camera(camera):

    camera = str(camera)

    if "|" in camera:

        front = camera.split("|")[1]

        values = re.findall(r"(\d+)MP", front)

        if values:
            return int(values[0])

    return np.nan

df["Front_MP"] = df["Camera"].apply(extract_front_camera)


In [21]:
df[
    [
        "Display_Size",
        "Display_Type",
        "Rear_Main_MP",
        "Front_MP"
    ]
].isnull().sum()

Display_Size     279
Display_Type       0
Rear_Main_MP      86
Front_MP        1098
dtype: int64

In [22]:
# ==========================================
# Improved Display Size Extraction
# ==========================================

def extract_display_size(display):
    display = str(display)

    # Case 1: Size mentioned in inches
    inch_match = re.search(r"\(([\d.]+)\s*inch", display, re.IGNORECASE)
    if inch_match:
        return float(inch_match.group(1))

    # Case 2: Only centimeters mentioned
    cm_match = re.search(r"([\d.]+)\s*cm", display, re.IGNORECASE)
    if cm_match:
        cm = float(cm_match.group(1))
        return round(cm / 2.54, 2)

    return np.nan


df["Display_Size"] = df["Display"].apply(extract_display_size)

In [23]:
# ==========================================
# Rear Main Camera Extraction
# ==========================================

def extract_rear_camera(camera):

    camera = str(camera)

    rear = camera.split("|")[0]

    values = re.findall(r"(\d+(?:\.\d+)?)MP", rear, re.IGNORECASE)

    if values:
        return float(values[0])

    return np.nan


df["Rear_Main_MP"] = df["Camera"].apply(extract_rear_camera)

In [24]:
# ==========================================
# Front Camera Extraction
# ==========================================

def extract_front_camera(camera):

    camera = str(camera)

    if "|" in camera:

        front = camera.split("|")[1]

        values = re.findall(r"(\d+(?:\.\d+)?)MP", front, re.IGNORECASE)

        if values:
            return float(values[0])

    return np.nan


df["Front_MP"] = df["Camera"].apply(extract_front_camera)


In [25]:
# ==========================================
# Rear Camera Count
# ==========================================

def rear_camera_count(camera):

    camera = str(camera)

    rear = camera.split("|")[0]

    return len(re.findall(r"(\d+(?:\.\d+)?)MP", rear))


df["Rear_Camera_Count"] = df["Camera"].apply(rear_camera_count)


In [26]:
# ==========================================
# Missing Value Treatment
# ==========================================

df["Display_Size"].fillna(df["Display_Size"].median(), inplace=True)

df["Rear_Main_MP"].fillna(df["Rear_Main_MP"].median(), inplace=True)

df["Front_MP"].fillna(df["Front_MP"].median(), inplace=True)


In [27]:
# ==========================================
# Engineered Features Summary
# ==========================================

engineered_cols = [
    "Display_Size",
    "Display_Type",
    "Rear_Main_MP",
    "Front_MP",
    "Rear_Camera_Count"
]

df[engineered_cols].isnull().sum()

Display_Size         0
Display_Type         0
Rear_Main_MP         0
Front_MP             0
Rear_Camera_Count    0
dtype: int64

In [28]:
import re

def extract_display_type(display):

    display = str(display).upper()

    if "DYNAMIC AMOLED" in display:
        return "Dynamic AMOLED"

    elif "SUPER AMOLED" in display:
        return "Super AMOLED"

    elif "AMOLED" in display:
        return "AMOLED"

    elif "POLED" in display:
        return "P-OLED"

    elif "OLED" in display:
        return "OLED"

    elif "IPS LCD" in display:
        return "IPS LCD"

    elif "IPS" in display:
        return "IPS"

    elif "LCD" in display:
        return "LCD"

    elif re.search(r'HD\+|FHD\+|FULL HD\+|FULL HD|HD DISPLAY', display):
        return "LCD (Unspecified)"

    else:
        return "Other"

df["Display_Type"] = df["Display"].apply(extract_display_type)

In [29]:
df.to_excel('Dataset1.xlsx', index=False)